In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm
from grid_cells.random_walk import generate_random_walk
from grid_cells.plot_tools import activity_map
from grid_cells.attractor_networks import ToroidBurakFiete2009
import os

DATA_DIR = "../simulation_data"
PLOTS_DIR = "../plots"

In [ ]:
T = 400
dt = 0.5e-3
n = 128
box_size = 3

net = ToroidBurakFiete2009(n=n, dt=dt, periodicity=19)
net.warm_up()
plt.imshow(net.s)

In [ ]:
position, velocity, time = generate_random_walk(T, dt, box_size)

n_steps = position.shape[0]
n_popul_snapshots = 1000
recorded_cells = list(np.random.randint(0, n - 1, size=(9, 2)))

recording = np.zeros((n_steps, len(recorded_cells)))
population_recordings = np.zeros((n_popul_snapshots, n, n))
popul_snapshots_integers = np.linspace(0, n_steps, n_popul_snapshots - 1, dtype=int)
snapshot_iter = 0
for step_iter in tqdm.tqdm(range(n_steps)):
    recording[step_iter] = np.array(
        [net.s[*cell_index] for cell_index in recorded_cells]
    )
    net.step(*velocity[step_iter])
    if step_iter == popul_snapshots_integers[snapshot_iter]:
        population_recordings[snapshot_iter] = net.s
        snapshot_iter += 1

In [ ]:
binned_activity, counts, edges = activity_map(position, recording[:, 0])
fig, ax = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": (1, 1.2)})
cb = ax[1].imshow(
    binned_activity.T,
    origin="lower",
    extent=[0, box_size, 0, box_size],
    cmap="jet",
    interpolation="bilinear",
)

ax[1].set_xlabel("x (m)")
ax[1].set_ylabel("y (m)")
ax[1].set_title("Firing Rate Map")
fig.colorbar(cb, ax=ax[1], label="Average Activity")

ax[0].set_title("Simulated Trajectory")
ax[0].plot(position[:, 0], position[:, 1], lw=0.5, color="gray")
ax[0].set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
ax[0].set_xlim((0, box_size))
ax[0].set_ylim((0, box_size))
fig.suptitle("Grid Cells Generated by Continuous Attractor Network")
fig.savefig(os.path.join(PLOTS_DIR, "can_grid_cells.png"))